In [21]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV, KFold
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored, integrated_brier_score
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.model_selection import StratifiedKFold


In [22]:
seed = 42
tau = 24
quick = True

R, K = (1, 5) if quick else (5, 5)
n_trees = 200 if quick else 500

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

cohort = "tcga"
data = pd.read_csv(f"data/{cohort}_clean.csv")
features = ["age", "male", "serosa", "n_level"]
data["male"] = (data["sex"] == "Male").astype(float)
X = data[features].astype(float)
y = Surv.from_arrays(event=data["event"].astype(bool).values, time=data["time_months"].values)
print(X.shape, "events:", int(data["event"].sum()))

(359, 4) events: 125


In [23]:
def c_index_tau(y_true, risk, tfe=tau):
    event_tfe = y_true["event"] & (y_true["time"] < tfe)
    return concordance_index_censored(event_tfe, y_true["time"], risk)[0]

def eval_times(y_test, tfe = tau):
    hi = min(tfe, y_test["time"].max() - 1e-6)
    lo = max(1.0, y_test["time"].min() + 1e-6)
    return np.linspace(lo, hi, 24)

def surv_at(model, X_, times):
    return np.vstack([fn(times) for fn in model.predict_survival_function(X_)])

def km_null_surv(y_train, times, n):
    t, s = kaplan_meier_estimator(y_train["event"], y_train["time"])
    s_at = np.array([s[t <= u][-1] if (t <= u).any() else 1.0 for u in times])
    return np.tile(s_at, (n, 1))

In [24]:
def time_of_eval_scorer(estimator, X_, y_):
    return c_index_tau(y_, estimator.predict(X_))

def inner_cv(y_tr):
    return list(StratifiedKFold(3, shuffle=True, random_state=seed).split(np.zeros(len(y_tr)), y_tr["event"]))

def make_cox(y_tr):
    pipe = Pipeline([("scale", StandardScaler()),
                     ("cox", CoxnetSurvivalAnalysis(fit_baseline_model=True))])
    grid = {"cox__alphas": [[1e-3], [1e-2], [1e-1], [1.0]], "cox__l1_ratio": [0.1, 0.5, 0.9]}
    return GridSearchCV(pipe, grid, cv=inner_cv(y_tr), scoring=time_of_eval_scorer, error_score=0.5)

def make_rsf(y_tr):
    rsf = RandomSurvivalForest(n_estimators=n_trees, random_state=seed, n_jobs=-1)
    grid = {"min_samples_leaf": [5, 10, 20, 40], "max_features": [1, 2, 4]}
    return GridSearchCV(rsf, grid, cv=inner_cv(y_tr), scoring=time_of_eval_scorer)

In [25]:
import copy
import numpy as np
import torch
from torch import nn
from sklearn.base import BaseEstimator
from sklearn.preprocessing import StandardScaler
from sksurv.functions import StepFunction
from sksurv.metrics import concordance_index_censored

In [26]:
class DeepSurv(BaseEstimator):
    def __init__(self, hidden_layers = 1, units = 16, dropout = 0.1, weight_decay = 1e-3, lr = 1e-3, max_epochs = 500, patience = 20, val_fraction = 0.2, n_seeds = 3, random_state = 0):
        self.hidden_layers = hidden_layers
        self.units = units
        self.dropout = dropout
        self.weight_decay = weight_decay
        self.lr = lr
        self.max_epochs = max_epochs
        self.patience = patience
        self.val_fraction = val_fraction
        self.n_seeds = n_seeds
        self.random_state = random_state

    def _build_net(self, n_inputs):
        layers, d = [], n_inputs
        for _ in range(self.hidden_layers):
            layers += [nn.Linear(d, self.units), nn.ReLU(), nn.Dropout(self.dropout)]
            d = self.units
        layers.append(nn.Linear(d, 1))
        return nn.Sequential(*layers)

    @staticmethod
    def _cox_loss(g, time, event):
        order = torch.argsort(time, descending=True)
        g, event = g[order], event[order]
        log_risk_set = torch.logcumsumexp(g, dim = 0)
        return -((g - log_risk_set) * event).sum() / event.sum().clamp(min = 1.0)

    def fit(self, X, y):
        X = np.asarray(X, dtype = np.float32)
        self.scaler_ = StandardScaler().fit(X)
        Xs = torch.tensor(self.scaler_.transform(X), dtype = torch.float32)
        time = torch.tensor(np.ascontiguousarray(y["time"], dtype=np.float32))
        event = torch.tensor(np.ascontiguousarray(y["event"], dtype=np.float32))

        rng = np.random.default_rng(self.random_state)
        ev = np.asanyarray(y["event"]).astype(bool)
        val = np.zeros(len(ev), dtype = bool)
        for grp in (np.where(ev)[0], np.where(~ev)[0]):
            k = max(1, int(round(self.val_fraction * len(grp))))
            val[rng.choice(grp, size = k, replace = False)] = True
        tr = ~val

        self.nets_ = []
        for s in range(self.n_seeds):
            torch.manual_seed(self.random_state + s)
            net = self._build_net(Xs.shape[1])
            opt = torch.optim.Adam(net.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            best_loss, best_state, wait = np.inf, None, 0

            for epoch in range(self.max_epochs):
                net.train()
                opt.zero_grad()
                loss = self._cox_loss(net(Xs[tr]).squeeze(1), time[tr], event[tr])
                loss.backward()
                opt.step()
                net.eval()
                with torch.no_grad():
                    val_loss = self._cox_loss(net(Xs[val]).squeeze(1), time[val], event[val]).item()
                if val_loss < best_loss - 1e-4:
                    best_loss, best_state, wait = val_loss, copy.deepcopy(net.state_dict()), 0
                else:
                    wait += 1
                    if wait >= self.patience:
                        break
            net.load_state_dict(best_state)
            net.eval()
            self.nets_.append(net)
        r = self._log_risk(X)
        t, e = np.asarray(y["time"], float), np.asarray(y["event"], bool)
        self.event_times_ = np.unique(t[e])
        exp_r = np.exp(r)
        H0 = np.cumsum([e[t == u].sum() / exp_r[t >= u].sum() for u in self.event_times_])
        self.baseline_cumhaz_ = H0
        return self

    def _log_risk(self, X):
        Xs = torch.tensor(self.scaler_.transform(np.asarray(X, dtype = np.float32)), dtype = torch.float32)
        with torch.no_grad():
            return np.mean([net(Xs).squeeze(1).numpy() for net in self.nets_], axis=0)

    def predict(self, X):
        return self._log_risk(X)

    def predict_survival_function(self, X):
        r = self._log_risk(X)
        x = np.r_[0.0, self.event_times_]
        return [StepFunction(x, np.exp(-np.r_[0.0, self.baseline_cumhaz_] * np.exp(ri))) for ri in r]

    def score(self, X, y):
        return concordance_index_censored(y["event"], y["time"], self.predict(X))[0]

In [27]:
def make_deepsurv(y_tr):
    net = DeepSurv(n_seeds = 3, random_state = seed)
    grid = {"hidden_layers": [1, 2], "units": [8, 32], "dropout": [0.1, 0.3], "weight_decay": [1e-4, 1e-2]}
    return GridSearchCV(net, grid, cv=inner_cv(y_tr), scoring=time_of_eval_scorer)

models = {"Cox": make_cox, "RSF": make_rsf, "DeepSurv": make_deepsurv}

In [17]:
rows, out_of_fold = [], []
cv = RepeatedStratifiedKFold(n_splits=K, n_repeats=R, random_state=seed)
t0 = time.time()

for split, (tr, te) in enumerate (cv.split(X, y["event"])):
    rep, fold = divmod(split, K)
    X_tr, X_te, y_tr, y_te = X.iloc[tr], X.iloc[te], y[tr], y[te]
    times = eval_times(y_te)

    risk0 = (X_te["serosa"] + X_te["n_level"]).values
    rows.append({"rep": rep, "fold": fold, "model": "Stage only", "c_tau": c_index_tau(y_te, risk0), "ibs": np.nan, "params": ""})
    rows.append({"rep": rep, "fold": fold, "model": "KM (no predictors)", "c_tau": 0.5,
                 "ibs": integrated_brier_score(y_tr, y_te, km_null_surv(y_tr, times, len(te)), times), "params": ""})

    for name, make in models.items():
        search = make(y_tr).fit(X_tr, y_tr)
        best = search.best_estimator_
        risk = best.predict(X_te)
        S = surv_at(best, X_te, times)
        rows.append({"rep": rep, "fold": fold, "model": name, "c_tau": c_index_tau(y_te, risk),
                     "ibs": integrated_brier_score(y_tr, y_te, S, times), "params": str(search.best_params_)})
        out_of_fold.append(pd.DataFrame({"rep": rep, "model": name, "patient_id": data["patient_id"].iloc[te].values,
                                 "risk": risk, "surv_tau": S[:, -1]}))
    print(f"split {split+1}/{R*K} done ({time.time()-t0:.0f}s)")

res = pd.DataFrame(rows)
out_of_fold = pd.concat(out_of_fold, ignore_index=True)
res.to_csv(results_dir / f"e1_{cohort}_folds.csv", index=False)
out_of_fold.to_csv(results_dir / f"e1_{cohort}_oof.csv", index=False)

C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

split 1/5 done (49s)


C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

split 2/5 done (97s)


C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

split 3/5 done (150s)


C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

split 4/5 done (226s)


C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

split 5/5 done (267s)


In [18]:
summary = res.groupby("model")[["c_tau", "ibs"]].agg(["mean", "std"]).round(3)
display(summary)

def corrected_ttest(d, n_train, n_test, alpha=0.05):
    d = np.asarray(d, float); J = len(d)
    se = np.sqrt((1 / J + n_test / n_train) * d.var(ddof=1))
    t = d.mean() / se
    p = 2 * stats.t.sf(abs(t), df=J - 1)
    q = stats.t.ppf(1 - alpha / 2, df=J - 1)
    return d.mean(), d.mean() - q * se, d.mean() + q * se, p

n_test = len(X) // K; n_train = len(X) - n_test
wide = res.pivot_table(index=["rep", "fold"], columns="model", values="c_tau")
EPS = 0.05
for m in ["RSF", "DeepSurv", "Stage only"]:
    d = wide[m] - wide["Cox"]
    mean, lo95, hi95, p = corrected_ttest(d, n_train, n_test)
    _, lo90, hi90, _ = corrected_ttest(d, n_train, n_test, alpha=0.10)
    equiv = (lo90 > -EPS) and (hi90 < EPS)
    print(f"{m} - Cox: {mean:+.3f}  95% CI [{lo95:+.3f}, {hi95:+.3f}]  p={p:.2f}  equivalent within ±{EPS}: {equiv}")

c_tau           ibs       
                     mean    std   mean    std
model                                         
Cox                 0.601  0.064  0.158  0.013
DeepSurv            0.604  0.066  0.157  0.011
KM (no predictors)  0.500  0.000  0.161  0.009
RSF                 0.592  0.076  0.160  0.014
Stage only          0.580  0.078    NaN    NaN

RSF - Cox: -0.009  95% CI [-0.059, +0.041]  p=0.65  equivalent within ±0.05: True
DeepSurv - Cox: +0.003  95% CI [-0.071, +0.077]  p=0.92  equivalent within ±0.05: False
Stage only - Cox: -0.022  95% CI [-0.109, +0.065]  p=0.52  equivalent within ±0.05: False


In [19]:
features = ["age", "male", "serosa", "n_level"]


def load_cohort(path):
    d = pd.read_csv(path)
    d["male"] = (d["sex"] == "Male").astype(float)
    X_ = d[features].astype(float)
    y_ = Surv.from_arrays(event=d["event"].astype(bool).values, time=d["time_months"].values)
    return d, X_, y_

cohorts = {"ACRG": load_cohort("data/acrg_clean.csv"),
           "TCGA": load_cohort("data/tcga_clean.csv")}
for name, (d, X_, y_) in cohorts.items():
    print(f"{name}: {X_.shape[0]} patients, {int(d['event'].sum())} events")

ACRG: 252 patients, 103 events
TCGA: 359 patients, 125 events


In [28]:
e2_rows, e2_preds, e2_params = [], {}, []
t0 = time.time()
for train_name, test_name in [("ACRG", "TCGA"), ("TCGA", "ACRG")]:
    direction = f"{train_name} -> {test_name}"
    _, X_train, y_train = cohorts[train_name]
    _, X_test, y_test = cohorts[test_name]
    times = eval_times(y_test)

    # baselines
    e2_preds[(direction, "Stage only")] = ((X_test["serosa"] + X_test["n_level"]).values, None)
    e2_preds[(direction, "KM (no predictors)")] = (np.zeros(len(X_test)), km_null_surv(y_train, times, len(X_test)))

    for name, make in models.items():
        search = make(y_train).fit(X_train, y_train)
        best = search.best_estimator_
        e2_preds[(direction, name)] = (best.predict(X_test), surv_at(best, X_test, times))
        e2_params.append({"direction": direction, "model": name, "params": str(search.best_params_)})
        print(f"{direction}: {name} done ({time.time() - t0:.0f}s)")

    for (dirn, m), (risk, S) in e2_preds.items():
        if dirn != direction:
            continue
        e2_rows.append({"direction": direction, "model": m,
                        "c_tau": c_index_tau(y_test, risk),
                        "ibs": integrated_brier_score(y_test, y_test, S, times) if S is not None else np.nan})

e2 = pd.DataFrame(e2_rows)
e2.to_csv(results_dir / "e2_cross_cohort.csv", index=False)
pd.DataFrame(e2_params).to_csv(results_dir / "e2_params.csv", index=False)
e2.round(3)

C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

ACRG -> TCGA: Cox done (0s)
ACRG -> TCGA: RSF done (11s)
ACRG -> TCGA: DeepSurv done (110s)


C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

TCGA -> ACRG: Cox done (110s)
TCGA -> ACRG: RSF done (121s)
TCGA -> ACRG: DeepSurv done (182s)


,direction,model,c_tau,ibs
0,ACRG -> TCGA,Stage only,0.573,NaN
1,ACRG -> TCGA,KM (no predictors),0.500,0.162
2,ACRG -> TCGA,Cox,0.592,0.168
3,ACRG -> TCGA,RSF,0.555,0.165
4,ACRG -> TCGA,DeepSurv,0.584,0.163
5,TCGA -> ACRG,Stage only,0.681,NaN
6,TCGA -> ACRG,KM (no predictors),0.500,0.164
7,TCGA -> ACRG,Cox,0.616,0.158
8,TCGA -> ACRG,RSF,0.592,0.164
9,TCGA -> ACRG,DeepSurv,0.595,0.162


In [29]:
B = 1000
rng = np.random.default_rng(seed)
summary = []
for direction in e2["direction"].unique():
    test_name = direction.split(" -> ")[1]
    _, _, y_test = cohorts[test_name]
    names = [m for (dirn, m) in e2_preds if dirn == direction]
    boot = {m: [] for m in names}
    for _ in range(B):
        idx = rng.integers(0, len(y_test), len(y_test))
        yb = y_test[idx]
        for m in names:
            boot[m].append(c_index_tau(yb, e2_preds[(direction, m)][0][idx]))
    boot = pd.DataFrame(boot)
    for m in names:
        point = e2.loc[(e2.direction == direction) & (e2.model == m), "c_tau"].item()
        lo, hi = np.percentile(boot[m], [2.5, 97.5])
        dlo, dhi = np.percentile(boot[m] - boot["Cox"], [2.5, 97.5])
        summary.append({"direction": direction, "model": m, "c_tau": point, "ci_low": lo, "ci_high": hi,
                        "diff_vs_cox": point - e2.loc[(e2.direction == direction) & (e2.model == "Cox"), "c_tau"].item(),
                        "diff_ci_low": dlo, "diff_ci_high": dhi})
e2_summary = pd.DataFrame(summary)
e2_summary.to_csv(results_dir / "e2_bootstrap.csv", index=False)
e2_summary.round(3)

,direction,model,c_tau,ci_low,ci_high,diff_vs_cox,diff_ci_low,diff_ci_high
0,ACRG -> TCGA,Stage only,0.573,0.521,0.624,-0.019,-0.036,-0.001
1,ACRG -> TCGA,KM (no predictors),0.500,0.500,0.500,-0.092,-0.144,-0.038
2,ACRG -> TCGA,Cox,0.592,0.538,0.644,0.000,0.000,0.000
3,ACRG -> TCGA,RSF,0.555,0.500,0.606,-0.037,-0.081,-0.000
4,ACRG -> TCGA,DeepSurv,0.584,0.530,0.635,-0.008,-0.024,0.007
5,TCGA -> ACRG,Stage only,0.681,0.629,0.735,0.065,0.009,0.127
6,TCGA -> ACRG,KM (no predictors),0.500,0.500,0.500,-0.116,-0.173,-0.051
7,TCGA -> ACRG,Cox,0.616,0.551,0.673,0.000,0.000,0.000
8,TCGA -> ACRG,RSF,0.592,0.532,0.647,-0.024,-0.060,0.011
9,TCGA -> ACRG,DeepSurv,0.595,0.528,0.656,-0.021,-0.041,-0.000


In [30]:
cv_mean = {c: pd.read_csv(results_dir / f"e1_{c.lower()}_folds.csv").groupby("model")["c_tau"].mean()
           for c in ["ACRG", "TCGA"]}
gap = e2[["direction", "model", "c_tau"]].copy()
gap["target_cohort"] = gap["direction"].str.split(" -> ").str[1]
gap["c_tau_cv_on_target"] = [cv_mean[t].get(m, np.nan) for t, m in zip(gap["target_cohort"], gap["model"])]
gap["transfer_gap"] = gap["c_tau"] - gap["c_tau_cv_on_target"]
gap.to_csv(results_dir / "e2_transfer_gap.csv", index=False)
gap.round(3)

,direction,model,c_tau,target_cohort,c_tau_cv_on_target,transfer_gap
0,ACRG -> TCGA,Stage only,0.573,TCGA,0.580,-0.006
1,ACRG -> TCGA,KM (no predictors),0.500,TCGA,0.500,0.000
2,ACRG -> TCGA,Cox,0.592,TCGA,0.601,-0.009
3,ACRG -> TCGA,RSF,0.555,TCGA,0.592,-0.038
4,ACRG -> TCGA,DeepSurv,0.584,TCGA,0.604,-0.020
5,TCGA -> ACRG,Stage only,0.681,ACRG,0.686,-0.005
6,TCGA -> ACRG,KM (no predictors),0.500,ACRG,0.500,0.000
7,TCGA -> ACRG,Cox,0.616,ACRG,0.680,-0.065
8,TCGA -> ACRG,RSF,0.592,ACRG,0.683,-0.091
9,TCGA -> ACRG,DeepSurv,0.595,ACRG,0.682,-0.087
